# 🛡️ Bug Bounty Copilot - Autonomous Training Pipeline (Kaggle T4 GPU)

This notebook trains an advanced offensive security reasoning model using **Unsloth (QLoRA 4-bit)** on **Qwen2.5-7B-Instruct**.
- **Dataset**: Curated PentesterLand Bug Bounty Chain-of-Thought (CoT) pairs.
- **Hardware**: Kaggle Free T4 GPU (Background Execution Supported).
- **Output**: LoRA adapter saved to `/kaggle/working/bugbounty_copilot_lora`.

In [ ]:
# 1. Install Unsloth & Core Dependencies (Optimized for T4 GPU)
!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers triton xformers


In [ ]:
# 2. Check for Previous Accumulated LoRA Weights (Mount & Resume)
from unsloth import FastLanguageModel
import torch, os, glob, urllib.request, json

max_seq_length = 4096
dtype = None # Auto detection (Float16 for T4)
load_in_4bit = True

# Injected by orchestrator before push (signed direct download URLs)
LATEST_LORA_URLS = {"bugbounty_copilot_lora/README.md": "https://www.kaggleusercontent.com/kf/358130309/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..qBw54ptmdEzwRXnzwvA3zA.lQu-Be6FcAfkl61xBT4TNAbfsmQzwomavU0IK_yKob169iTUIF7FGHJJ6fOPVQZP7Swq5oqRqeuNZXiOtM7QKyR51nb2M5vqtY9E1BhGTGvkaVz79dRuUsqGdWPomFYFfKlQbZdAnpNmfaDO4dxM7mEtIfhkY2gH9ZGQ7CYrrua82wTB-c4ZAc6CKwggqM3OMDEgNyT9Fjqauoz-WGRB0t0MWIVfjGVD1c8onwyXUz9CbYXbS8ISJgMaMB83czo7YOH8lOewqAmOlXTypajTidBN-cnzMZqfY4Ed4V1pPn-qvTxL_4of1h8K0C9fxiKXz5qqhjl_JBCc7LFe9fb8N7sghObtiSv1IAfDUOmmUPGQ04XNLeXcz4WBroKsUHeneNreFNXNt_oCYtVNA_S7DwdSuTPaXfPAcbr_74ZnUN9MScCi0RliRuPpekF8fgxjP4jGRdrdQi_EDpAcJB6aELoOUm0QxeCQCeQyx0gGxNj2KerTyvHpERJGhCBMga1oSBCkmFK-BuXosRl1d1fh2IWKJ7ySWKrqLduh2wYP5DIA-6o5P9OG9HEr9Rqbf8F6lX_nzK1R_ha-8x4zAMv_34_tE8BGVeqiIbqDMUuUmc6D9dlYV0kTheEzBIKTQLsb.3KUgZW23c-dOiAKea4yjyA/bugbounty_copilot_lora/README.md", "bugbounty_copilot_lora/adapter_config.json": "https://www.kaggleusercontent.com/kf/358130309/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..pEer1bIr9-JEz4MhCAHqEQ.8hfymw0Bb3DqMq26BbxhnTHZwrMlhiCOP--U7wpcOk8x_52x34o8Ws1OcJH-3Eftft8Tv5D92WtdcGlRTZyZe4AJQdFYWElUOvIv9NwNbrq3CtCixFUurj-YbXnqs7g_ymAc4NbNnuXsrdDlfetkZkJsmVI1c5srUJ44VZ8eYUQS4h2jw7Q1BpRUIaMasqU7Ag5EK_ibOAy_pviMmkKxgCSJ-BFmDV6u2q7J3UFl_SzK3ZW4z0wq1Ddh5chcJ72hbV_PapbC1ZAYPOlF6JOD0qKqN4J3E11sky-tekmhfToVsy53m1E-s57EFAc0UKSYCmn7gPZ5tAkrEhtXFrdGhqqvjROFLdcu8LuGvZN4fG9mWs0A3EvnK0rsu44uEH-8gqH8hjFT_Ds9k-R-Mp_Jt0znwf2Rj_VmMTeUGFhBs9z9G-dxZTLsh5nFBW5xdn_pHTckqRwqbhtxlwWfw8Arhlfd3YDwfdaCx-opa_-ob-lr-SvkJm3sNhkG3hpOlTelkTsPhUUMpZUVQgZRXCezpLK9_yjLSb2iS7iKO-Ho2z5Qwp_jsNIDby9CBU1LCf5KTGQyhLOcC3J5l5hEyCb9YfFWsDD4F3AsPb4iUvSrtJkGUnilubFuAGHU3sIbSaUh.HbPcZeWN_COHTKG6bYtWbw/bugbounty_copilot_lora/adapter_config.json", "bugbounty_copilot_lora/adapter_model.safetensors": "https://www.kaggleusercontent.com/kf/358130309/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..Ulw3vdlHM0W3Fah9_rVlqw.R2Sxnv7wd9bSxW6xT0kDaGlbeg5g4HwWC50P7_ZS2NwyKXlQDbyIxu6vJkpvMJDFhp3BJMNbM9kEAuAKOFPsCY6x5MpyYk2E_Y5anBzpwO2QSmMkZ8_TyHmY64Od-IdN08sZG64oufR-VkAdAcPhCFuTGEWUAxQ0pZHWqERbBaPmYKvVifp8KKDmkjFtYXqTKfS2UeRzlF4e_yEYeY4ZHjLfwiEPBpXLTTfRQGXLWVK-6FOfaK0nZ6dqsYOQIs8Wm1oLu0BMS8Jp1RMbYKyIoCGoOQ5zshZb5gf4MrqVx2kMCAw-tmPWdW9QBLoxf4gINwJGB-I_fArZuFp3NmmbMTJFVDK1D9cGt96irBaBer8occsx58cFT-m7a0VKplQYQ1IUtnGKMD07e2p8QbXm_up6hOYEGMf48tCbX1aBjzt2dpinAz381o5QCXmALAPM6PyY-JwIhhQm4E2rY16q0FIP27d8RZBNvTcXi2IFNdHtv81BUNR1tJQm0so-_oOX3tj1p5jhEP65otqXAWEdpLQiYgz0yk-QIrxlNOxzeIODMAVBxZ0IPs7cMmXTYKVaNZwUq8ejmNo3pOU1bNHAUz3PNM2nCl_Fse_PqBN2N_NAS-TdJo3RY_4kZCmNHqpe.jaAvd-YqIwiTHCOMbpFBUg/bugbounty_copilot_lora/adapter_model.safetensors", "bugbounty_copilot_lora/chat_template.jinja": "https://www.kaggleusercontent.com/kf/358130309/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..ij268nrpU-MLqsomjPLFHg.CGRImh2D7UUuYW2KVj0eSqPFEpUjZltHxFw3w28aUDSfUGJ0zxrAmzexo9ceDA_7QFRRlLrzd7CqGWmBzaovHbMJhc109Odw1dMtHeuSQZcKRInlTRKz77KYbuUmF9BT_qpigHzFNDPk5Iu-7AHdBPYzsux0Gig3-TICuFO5e_z9Mr-jU-sJH3O22h24bQ81f79BHiQ8GSMps3KvLh3X9Fk9aKhUDK2cT9EM8owqM99KWiMbMlWMWNpzJHeGSHK5_enUz6t2DW9b6NIRZjMasosjPYWfd8r-galg5QganC3EWFT_PU-cENuCwGTyB9aTdg3U7f-WwCspIyfA959WQXCcpp-0GTMEncY65aTuSEnjNzMO0ZHIyzI52YfPZC3KetPqamC23m-F2iQDoxEZkE6AdiziGifU2uoICoXgqewTINi7TigTGFjbrRCWqpK9ZtzS7aJSk4OlmSH0qXaynu-ZL5K9-XBjthcjm_3AbAWo9cNRv9oQoM-CQoQr7qdwWbsomb9vOOi1hf11IJwRn4uCRZItgOD1-qlTyPGKaBQtjmYIcx5sewk_hctLR4bwPnN0HPxGR8w3JdWCZp8ddbyq31gtgFpYtl0mRMKzv9oELDHr_H4z7MsiZS4_wFtA.MNlkFcuO4DJkf7_ZNCeMSg/bugbounty_copilot_lora/chat_template.jinja", "bugbounty_copilot_lora/tokenizer.json": "https://www.kaggleusercontent.com/kf/358130309/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..mzAMW3ntm37DAh3knKL6lw.39RoyO-w6b20lN9GPNOHfa5mWksnDk2AEOpgIlJN40CsR5iF8tbRlUaB1Mb9ittkgN8V7Nc_oo2Rb3Hi1iDaLX1vaFhe1ZWhgEoee87s3xrcF2Nxx_qiAgNJdLClrLakPL1jhEtYEqy-IgdCsvoiIHhkSG3xB74DXrKQ0m1pjUixAO191Wl1oI7AtsM8locH2Mu1EXjEX0zA23epS66mjfMFtOE-dGN4JZOTzzG37-6VnqbTWSybcUAK3qnOU_OZkGveQzuRbu7mo6PK67jIhPURyejq4Dk6wHUofrYISvqmmdDOH2ZJ9DT6iYRc-AGqE4T-nE8PpMnYkCeW-yDZglGkBkV0J5j_nn_d9xHsCikmvxy8eWdV-0525C9tUEZLdVLe0Jcldf1oqCnuWJgpvnbzWVI1PWicbbC--_HcXbTwJJoeerB8lOj_YA79YHcgagfM5qLSaDkJMegORZ0oIfEPGZ10qitBQCv-q9kVsESWoHJlV5GSTb9T2rjVPlFjMET-SDFDqEPnG066LuocsitIYcGaK0JihdfJkoHOPAcepJYmWOp55ndgNtj520Qcrate2hxbIQfMG8hhYSC-z6eHXi_lTpVbdHHqFe2kHxdPwXEXQ3MzeGlLSVMUb-w_.1isozVgmkVBiT3BTpSoWaQ/bugbounty_copilot_lora/tokenizer.json", "bugbounty_copilot_lora/tokenizer_config.json": "https://www.kaggleusercontent.com/kf/358130309/eyJhbGciOiJkaXIiLCJlbmMiOiJBMTI4Q0JDLUhTMjU2In0..VFpFE6OdtXHxBxE6yjF_zw.5iRrsWg90fg13aKRPt4C1oseyIHLpepb5A5KHChs7cS9s3TzqT7uq8JzHza-96YoDD3rTMM96pO9WsDWomzeO8iFAEIZR-DZTur7fQXlvOcOdGJlAcwd692KEvyIvYrKwJvjAtVNSmwXC962okEfyr9pM-q52pFJjPSkx1N18QcT23yhKrGKuKR3-5GvkEj4QyOK02epgR2__dXeKLV44OfQnNFdVXNc3sMp65ar9Vvk_IGJ-YGvH92-o3H2qtMgtcW_ncobcbIYd4y8q1cGo4lEchRCPL2onUHewQQBE-v90MUmpix43EE0-GyvmOcg-aYhgfRxGTbhsQxhimwhsprG17dfy40OVqCVe-TQzxSekpSNxVttSrFfV69puoSDpQdvqsh5S1vEuFXm22SXNyc1WLDrLgCeHsQT9BzhNJutMjF8AR6yqdpgAHF-etwsMY08W6YKwQBzoaKHagP8b8STRRfJIeoSVF49FZNaVHeMffY-IIJkf_MmjMsJ4GMoBYaQdE2UQC_0pJ28I8InahVx-7c4L5G1UXo-CnD94GuluyVZzJ1gzZby2hbNilc9ZkIV4rSsctT8y7nybKx_a69iAOg7wj2sM-OpCgS8vNB6YrUXm3agm-C94CmHkDkZ.WtBGA7na4vJNrij7jconZg/bugbounty_copilot_lora/tokenizer_config.json"} # Daemon injected signed URLs

# Priority 1: Check mounted kernel sources in /kaggle/input/
possible_loras = glob.glob("/kaggle/input/**/bugbounty_copilot_lora/adapter_model.safetensors", recursive=True)
if not possible_loras:
    possible_loras = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)

# Priority 2: Direct signed URL download from partner node
if not possible_loras and LATEST_LORA_URLS:
    print(f"[*] Downloading {len(LATEST_LORA_URLS)} accumulated LoRA files from previous run...")
    lora_dir = "/kaggle/working/latest_accumulated_lora"
    os.makedirs(lora_dir, exist_ok=True)
    for fname, url in LATEST_LORA_URLS.items():
        local_f = os.path.join(lora_dir, os.path.basename(fname))
        try:
            urllib.request.urlretrieve(url, local_f)
            print(f"    -> Downloaded: {os.path.basename(fname)}")
        except Exception as e:
            print(f"    -> Warning downloading {fname}: {e}")
    if os.path.exists(os.path.join(lora_dir, "adapter_model.safetensors")):
        possible_loras = [os.path.join(lora_dir, "adapter_model.safetensors")]

# Priority 3: Fallback via CLI
if not possible_loras:
    print("[*] Searching for partner node weights via Kaggle direct output pull...")
    os.system("kaggle kernels output mmossad2324/bugbounty-copilot-trainer -p /kaggle/working/prev_acc3 >/dev/null 2>&1 || true")
    os.system("kaggle kernels output mmossad2124/bugbounty-copilot-trainer -p /kaggle/working/prev_acc1 >/dev/null 2>&1 || true")
    candidates = glob.glob("/kaggle/working/prev_acc*/**/adapter_model.safetensors", recursive=True)
    if candidates:
        candidates.sort(key=lambda x: os.path.getmtime(x), reverse=True)
        possible_loras = [candidates[0]]

loaded_from_previous = False
if possible_loras:
    prev_adapter_dir = os.path.dirname(possible_loras[0])
    print(f"[✓] FOUND PREVIOUS DAY ACCUMULATED WEIGHTS at: {prev_adapter_dir}")
    print("[*] Resuming and stacking incremental training on top of previous day progress...")
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = prev_adapter_dir,
            max_seq_length = max_seq_length,
            dtype = dtype,
            load_in_4bit = load_in_4bit,
        )
        FastLanguageModel.for_training(model)
        loaded_from_previous = True
        print("[✓] Successfully loaded previous day accumulated weights! Knowledge is compounding.")
    except Exception as e:
        print(f"[!] Could not load previous weights directly ({e}). Falling back to base model.")

if not loaded_from_previous:
    model_name = "Qwen/Qwen2.5-7B-Instruct"
    print(f"[*] Initializing fresh base model: {model_name} in 4-bit...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = model_name,
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = 16,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj"],
        lora_alpha = 16,
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
        use_rslora = False,
        loftq_config = None,
    )
    print("[✓] Attached fresh QLoRA adapters to base model!")


In [ ]:
# 3. Verify Active LoRA Training Status
FastLanguageModel.for_training(model)
print('[✓] Model & LoRA Adapters verified in active training mode!')


In [ ]:
# 4. Load & Format Training Dataset (ChatML Format with Industrial Conveyor Specialization)
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml", # Qwen2.5 standard
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

# Dynamic Relay Specialization with Industrial Conveyor Chunks:
ACTIVE_NODE_USER = "mmossad2124" # Daemon auto-updated

# Auto-detect node identity from mounted input if available
if any("mmossad2124" in p for p in glob.glob("/kaggle/input/*")):
    ACTIVE_NODE_USER = "mmossad2324"
elif any("mmossad2324" in p for p in glob.glob("/kaggle/input/*")):
    ACTIVE_NODE_USER = "mmossad2124"

if ACTIVE_NODE_USER == "mmossad2324":
    dataset_file = "active_stage2_chunk.jsonl"
    print("[*] Node Specialization: Account 3 (Stage 2: Kernel, Memory, APTs Active Chunk - 2,400 samples)")
else:
    dataset_file = "active_stage1_chunk.jsonl"
    print("[*] Node Specialization: Account 1 (Stage 1: Web, Cloud, AI Active Chunk - 2,400 samples)")

dataset_url = f"https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets/{dataset_file}"
print(f"[*] Loading conveyor chunk from: {dataset_url}")
try:
    dataset = load_dataset("json", data_files=dataset_url, split="train")
except Exception as e:
    print(f"[!] Active chunk notice ({e}). Falling back to full dataset file...")
    fallback_file = "stage2_kernel_apt_arsenal.jsonl" if ACTIVE_NODE_USER == "mmossad2324" else "stage1_web_cloud_arsenal.jsonl"
    dataset = load_dataset("json", data_files=f"https://raw.githubusercontent.com/mmossad2124-blip/BugBounty-Copilot/main/data_pipeline/processed_datasets/{fallback_file}", split="train")

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"[✓] Successfully loaded {len(dataset)} elite offensive reasoning samples!")


In [ ]:
# 5. Configure SFTTrainer (Supervised Fine-Tuning)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/kaggle/working/checkpoints",
        save_steps = 100,
    ),
)
print("[✓] Trainer configured and ready!")

In [ ]:
# 6. Train the Model!
print("[*] Starting training loop on GPU...")
trainer_stats = trainer.train()
print("[✓] Training completed successfully!")

In [ ]:
# 7. Test Inference (Sample Vulnerability Reasoning)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are an Elite Offensive Security & Bug Bounty AI Copilot."},
    {"role": "user", "content": "Target scope: api.target.com/v1/user/profile. How would you methodically test for IDOR and privilege escalation?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

In [ ]:
# 8. Save LoRA Adapters to Kaggle Output
output_dir = "/kaggle/working/bugbounty_copilot_lora"
print(f"[*] Saving trained LoRA adapter to {output_dir}...")
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("[✓] LoRA adapters permanently saved to output! Available in Kaggle output tab.")